# Seq2Seq con redes recurrentes: traducción inglés → francés

En el laboratorio anterior usamos una RNN/LSTM para **clasificar** una secuencia (identificar su idioma): la entrada era una secuencia y la salida, una sola etiqueta. Ahora la salida también es una **secuencia de largo distinto** y sin alineación palabra a palabra: traducir `go .` → `va !`.

Para esto construiremos un modelo **seq2seq** (*sequence to sequence*) con la arquitectura **codificador–decodificador** de Sutskever et al. (2014) y Cho et al. (2014):

* El **codificador** (una LSTM) lee la oración en inglés y la resume en un estado final $(\mathbf{H},\mathbf{C})$ (la *variable de contexto*).
* El **decodificador** (otra LSTM) parte de ese estado y genera la oración en francés un token a la vez.

El diseño sigue el artículo [*Building a Plain Seq2Seq Model for Language Translation*](https://machinelearningmastery.com/building-a-plain-seq2seq-model-for-language-translation/) de Adrian Tam (Machine Learning Mastery): mismo dataset (pares inglés–francés de Anki/Tatoeba), tokenizadores BPE separados para cada idioma, codificador y decodificador LSTM de 2 capas con embeddings y estados ocultos de 256 dimensiones. Las **diferencias** respecto del artículo, que corresponden a lo visto en clase, son:

| Artículo | Este laboratorio |
|---|---|
| Entrena sobre todos los datos y "evalúa" sobre los mismos datos | Separa un 10% de **pares de prueba** y mide la pérdida y BLEU en ellos |
| El decodificador vuelve a procesar el prefijo completo en cada paso y usa sus propias predicciones durante el entrenamiento | **Teacher forcing**: el decodificador recibe el objetivo real desplazado un token; en inferencia avanza **un token por paso**, guardando su estado |
| Sin recorte de gradientes, capa de salida de tamaño `embedding_dim` | Recorte de gradientes y capa de salida de tamaño `hidden_dim` |
| Solo búsqueda voraz | Búsqueda voraz, **búsqueda en haz** y métrica **BLEU** |
| Solo seq2seq simple | Además, el decodificador con **atención aditiva de Bahdanau** |

> **Requisitos:** `torch`, `tokenizers`, `matplotlib`, `pandas` y conexión a internet para descargar el dataset la primera vez. Una GPU acelera bastante el entrenamiento.

In [ ]:
import os
import math
import random
import zipfile
import unicodedata
import collections
import urllib.request

import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from tokenizers import Tokenizer, models, pre_tokenizers, decoders, trainers

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Usando {device}")

random.seed(42)
torch.manual_seed(42)

## El dataset

Usamos los pares de oraciones inglés–francés del [proyecto Anki/Tatoeba](https://www.manythings.org/anki/). El archivo `fra.txt` tiene una oración en inglés y su traducción separadas por un tabulador (algunas versiones agregan una tercera columna de atribución, que ignoramos).

El preprocesamiento es el del artículo: normalización Unicode NFKC (que, por ejemplo, convierte los espacios no separables en espacios normales) y paso a minúsculas. Luego separamos el **10% de los pares** para prueba, de modo que ninguna oración de prueba aparezca en el entrenamiento.

In [ ]:
DATA_URL = 'http://storage.googleapis.com/download.tensorflow.org/data/fra-eng.zip'

def download(url=DATA_URL, root='./data'):
    os.makedirs(root, exist_ok=True)
    fname = os.path.join(root, 'fra-eng-anki.zip')
    if not os.path.exists(fname):
        print(f"Descargando {url}...")
        urllib.request.urlretrieve(url, fname)
    return fname

def normalize(line):
    return unicodedata.normalize('NFKC', line.strip().lower())

text_pairs = []
with zipfile.ZipFile(download()) as z:
    with z.open('fra.txt') as f:
        for line in f.read().decode('utf-8').splitlines():
            parts = normalize(line).split('\t')
            if len(parts) >= 2:
                text_pairs.append((parts[0], parts[1]))

random.shuffle(text_pairs)
n_test = len(text_pairs) // 10
test_pairs, train_pairs = text_pairs[:n_test], text_pairs[n_test:]
print(f"{len(text_pairs)} pares: {len(train_pairs)} de entrenamiento, {len(test_pairs)} de prueba")
for en, fr in train_pairs[:3]:
    print(f"{en}  ->  {fr}")

## Tokenizadores BPE

Como el artículo, usamos **un tokenizador por idioma** de tipo *byte-level BPE* (codificación por pares de bytes) con vocabulario de 8.000 tokens. Así evitamos tener palabras desconocidas y mantenemos vocabularios pequeños (el dataset tiene unas 15.000 palabras distintas en inglés y 30.000 en francés). Reservamos tres tokens especiales:

* `[start]` y `[end]` marcan el inicio y el fin de la oración de salida (en clase: `<bos>` y `<eos>`).
* `[pad]` rellena las secuencias de un minibatch hasta el mismo largo.

Los tokenizadores se entrenan **solo con los pares de entrenamiento**.

In [ ]:
VOCAB_SIZE = 8000
SPECIAL = ['[start]', '[end]', '[pad]']

def train_tokenizer(texts, vocab_size=VOCAB_SIZE):
    tok = Tokenizer(models.BPE())
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)
    tok.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=SPECIAL,
                                  initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
                                  show_progress=False)
    tok.train_from_iterator(texts, trainer=trainer)
    return tok

en_tok = train_tokenizer(en for en, fr in train_pairs)
fr_tok = train_tokenizer(fr for en, fr in train_pairs)

START, END, PAD = (fr_tok.token_to_id(t) for t in SPECIAL)
EN_PAD = en_tok.token_to_id('[pad]')
print("Vocabularios:", en_tok.get_vocab_size(), fr_tok.get_vocab_size())

en, fr = random.choice(train_pairs)
print(en, '->', en_tok.encode(en).tokens)
print(fr, '->', fr_tok.encode(fr).tokens)
print(fr_tok.decode(fr_tok.encode(fr).ids))

## Minibatches

Cada ejemplo es un par (oración en inglés, oración en francés). La función `collate` convierte los textos en índices y rellena con `[pad]`:

* la **entrada del codificador** `src` (con sus largos reales, que usaremos para empaquetar la secuencia, como en el laboratorio 3.2);
* el objetivo `tgt = [start] + francés + [end]`. De él se obtienen la **entrada del decodificador** (`tgt[:, :-1]`, que empieza con `[start]`) y las **etiquetas** (`tgt[:, 1:]`, el mismo texto desplazado un token y terminado en `[end]`). Es el *teacher forcing*.

Truncamos las oraciones a `MAX_LEN` tokens.

In [ ]:
MAX_LEN = 40
BATCH_SIZE = 64

def collate(batch):
    src = [torch.tensor(en_tok.encode(en).ids[:MAX_LEN]) for en, fr in batch]
    tgt = [torch.tensor([START] + fr_tok.encode(fr).ids[:MAX_LEN] + [END]) for en, fr in batch]
    lengths = torch.tensor([len(s) for s in src])
    src = pad_sequence(src, batch_first=True, padding_value=EN_PAD)
    tgt = pad_sequence(tgt, batch_first=True, padding_value=PAD)
    return src, lengths, tgt

train_loader = torch.utils.data.DataLoader(train_pairs, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
test_loader = torch.utils.data.DataLoader(test_pairs, batch_size=256, collate_fn=collate)

src, lengths, tgt = next(iter(train_loader))
print("src:", src.shape, "largos:", lengths[:5].tolist(), "tgt:", tgt.shape)
print("entrada del decodificador:", [fr_tok.id_to_token(i) for i in tgt[0, :-1].tolist()])
print("etiquetas:               ", [fr_tok.id_to_token(i) for i in tgt[0, 1:].tolist()])

## El codificador

Una capa `Embedding` seguida de una LSTM multicapa. Empaquetamos la secuencia (`pack_padded_sequence`) para que el relleno no contamine el estado final: si no lo hiciéramos, el estado final $(\mathbf{H},\mathbf{C})$ habría pasado por pasos de relleno después de terminar la oración.

El codificador devuelve:
* `outputs`: el estado oculto de la última capa en cada paso, forma `(B, T, hidden)`. Solo los usará el modelo con atención.
* `(h, c)`: los estados finales de todas las capas, forma `(num_layers, B, hidden)`. Este par es la **variable de contexto**.

In [ ]:
class EncoderLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers=2, dropout=0.1, pad_id=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_id)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, num_layers, batch_first=True,
                           dropout=dropout if num_layers > 1 else 0)

    def forward(self, src, lengths):
        x = self.embedding(src)                                   # (B, T, E)
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, (h, c) = self.rnn(packed)
        outputs, _ = pad_packed_sequence(out, batch_first=True, total_length=src.size(1))
        return outputs, (h, c)                                    # (B, T, H), (L, B, H)

## El decodificador y el modelo seq2seq

El decodificador tiene la misma estructura (embedding + LSTM) más una capa lineal que produce los *logits* sobre el vocabulario francés. Su estado inicial es el estado final del codificador, por lo que ambos deben tener el mismo número de capas y tamaño oculto.

`Seq2SeqLSTM` expone dos interfaces:

* `forward(src, lengths, tgt_in)`: para **entrenar**, con teacher forcing, procesa toda la secuencia objetivo de una vez.
* `start(src, lengths)` y `step(y_prev, state)`: para **inferir**, `start` codifica la entrada y `step` avanza **un token** del decodificador, devolviendo los logits y el nuevo estado. Las funciones de traducción de más abajo solo usan esta interfaz.

In [ ]:
class DecoderLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers=2, dropout=0.1, pad_id=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_id)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, num_layers, batch_first=True,
                           dropout=dropout if num_layers > 1 else 0)
        self.out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, tgt_in, state):                  # tgt_in: (B, T')
        o, state = self.rnn(self.embedding(tgt_in), state)
        return self.out(o), state                      # (B, T', V)


class Seq2SeqLSTM(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder, self.decoder = encoder, decoder

    def forward(self, src, lengths, tgt_in):           # entrenamiento (teacher forcing)
        _, state = self.encoder(src, lengths)
        logits, _ = self.decoder(tgt_in, state)
        return logits

    def start(self, src, lengths):                     # inferencia: codificar
        _, state = self.encoder(src, lengths)
        return state

    def step(self, y_prev, state):                     # inferencia: un token (B,)
        logits, state = self.decoder(y_prev.unsqueeze(1), state)
        return logits[:, 0], state, None

In [ ]:
EMB_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT = 256, 256, 2, 0.1

def build_seq2seq():
    enc = EncoderLSTM(en_tok.get_vocab_size(), EMB_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT, EN_PAD)
    dec = DecoderLSTM(fr_tok.get_vocab_size(), EMB_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT, PAD)
    return Seq2SeqLSTM(enc, dec).to(device)

model = build_seq2seq()
print(model)
print(f"Parámetros entrenables: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# Comprobación de formas
src, lengths, tgt = next(iter(train_loader))
logits = model(src.to(device), lengths, tgt[:, :-1].to(device))
print("logits:", tuple(logits.shape), "= (batch, pasos del decodificador, vocabulario)")

## Entrenamiento con pérdida enmascarada

Entrenamos con entropía cruzada **ignorando las posiciones de relleno** (`ignore_index=PAD`): el promedio se calcula solo sobre los tokens reales, es decir $\mathcal{L}=\sum_i m_i\ell_i / \sum_i m_i$ con $m_i = \mathbb{1}[y_i \neq \texttt{[pad]}]$. Usamos Adam con tasa de aprendizaje $10^{-3}$ y **recorte de gradientes** a norma 1.

Al terminar cada época evaluamos la pérdida y la **perplejidad** ($\exp$ de la pérdida) sobre el conjunto de prueba. Para acortar el tiempo de entrenamiento usamos pocas épocas (`EPOCHS`); súbelas si tienes tiempo.

In [ ]:
EPOCHS = 6
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD)

@torch.no_grad()
def evaluate_loss(model, loader):
    model.eval()
    total, tokens = 0.0, 0
    for src, lengths, tgt in loader:
        src, tgt = src.to(device), tgt.to(device)
        logits = model(src, lengths, tgt[:, :-1])
        labels = tgt[:, 1:]
        total += nn.functional.cross_entropy(logits.reshape(-1, logits.size(-1)), labels.reshape(-1),
                                             ignore_index=PAD, reduction='sum').item()
        tokens += (labels != PAD).sum().item()
    return total / tokens

def fit(model, epochs=EPOCHS, lr=1e-3, clip=1.0, report_freq=500):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        running, n = 0.0, 0
        for i, (src, lengths, tgt) in enumerate(train_loader, 1):
            src, tgt = src.to(device), tgt.to(device)
            optimizer.zero_grad()
            logits = model(src, lengths, tgt[:, :-1])
            loss = loss_fn(logits.reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), clip)
            optimizer.step()
            running += loss.item(); n += 1
            if i % report_freq == 0:
                print(f"  época {epoch}, minilote {i}: pérdida = {running / n:.4f}")
        test_loss = evaluate_loss(model, test_loader)
        history.append((running / n, test_loss))
        print(f"época {epoch}/{epochs}: pérdida entrenamiento = {running / n:.4f}, "
              f"pérdida prueba = {test_loss:.4f}, perplejidad prueba = {math.exp(test_loss):.2f}")
    return history

hist_plain = fit(model)

## Traducir: búsqueda voraz

Para traducir una oración nueva:
1. Codificamos la entrada y obtenemos el estado inicial del decodificador.
2. Alimentamos `[start]` en el primer paso.
3. En cada paso elegimos el token de mayor probabilidad (`argmax`) y lo usamos como entrada del paso siguiente.
4. Paramos al emitir `[end]` o al llegar a `MAX_LEN`.

Las funciones usan solo `model.start` y `model.step`, así que más adelante funcionarán también con el modelo con atención.

In [ ]:
def encode_source(sentence):
    ids = en_tok.encode(normalize(sentence)).ids[:MAX_LEN]
    return torch.tensor([ids], device=device), torch.tensor([len(ids)])

@torch.no_grad()
def translate_greedy(model, sentence, max_len=MAX_LEN):
    model.eval()
    src, lengths = encode_source(sentence)
    state = model.start(src, lengths)
    y = torch.tensor([START], device=device)
    out = []
    for _ in range(max_len):
        logits, state, _ = model.step(y, state)
        y = logits.argmax(-1)
        if y.item() == END:
            break
        out.append(y.item())
    return fr_tok.decode(out).strip()

for en, fr in random.sample(test_pairs, 8):
    print(f"EN : {en}\nFR : {fr}\nMOD: {translate_greedy(model, en)}\n")

## Búsqueda en haz (*beam search*)

La búsqueda voraz no es óptima: elegir lo mejor en cada paso no garantiza la mejor **secuencia** completa. La búsqueda exhaustiva sí lo es, pero cuesta $\mathcal{O}(|\mathcal{Y}|^{T'})$. La **búsqueda en haz** mantiene los `k` mejores candidatos parciales: en cada paso extiende cada uno con sus `k` mejores tokens y conserva los `k` de mayor probabilidad conjunta. Cuesta $\mathcal{O}(k|\mathcal{Y}|T')$ y con $k=1$ es la búsqueda voraz.

Al final elegimos entre los candidatos terminados con la puntuación normalizada por el largo

$$\frac{1}{L^{\alpha}}\log P(y_1,\ldots,y_L\mid\mathbf{c}),\qquad \alpha = 0.75,$$

porque sin normalizar se favorecen las secuencias cortas (cada token agrega un $\log P<0$).

In [ ]:
@torch.no_grad()
def translate_beam(model, sentence, k=4, alpha=0.75, max_len=MAX_LEN):
    model.eval()
    src, lengths = encode_source(sentence)
    state = model.start(src, lengths)
    beams = [(0.0, [START], state)]       # (log prob, tokens, estado)
    finished = []
    for _ in range(max_len):
        candidates = []
        for logp, seq, st in beams:
            logits, new_st, _ = model.step(torch.tensor([seq[-1]], device=device), st)
            logprobs = torch.log_softmax(logits[0], dim=-1)
            top = logprobs.topk(k)
            for v, i in zip(top.values.tolist(), top.indices.tolist()):
                candidates.append((logp + v, seq + [i], new_st))
        candidates.sort(key=lambda c: c[0], reverse=True)
        beams = []
        for cand in candidates[:k]:
            (finished if cand[1][-1] == END else beams).append(cand)
        if not beams:
            break
    finished += beams                     # candidatos que no llegaron a [end]
    best = max(finished, key=lambda c: c[0] / (len(c[1]) - 1) ** alpha)
    return fr_tok.decode([t for t in best[1][1:] if t != END]).strip()

for en, fr in random.sample(test_pairs, 5):
    print(f"EN : {en}\nFR : {fr}\nvoraz: {translate_greedy(model, en)}\nhaz  : {translate_beam(model, en, k=4)}\n")

## Evaluación con BLEU

Comparar oraciones a mano no escala. **BLEU** compara los $n$-gramas de la predicción con los de la referencia:

$$\mathrm{BLEU} = \exp\Big(\min\big(0,\,1-\tfrac{\mathrm{len}_{\mathrm{ref}}}{\mathrm{len}_{\mathrm{pred}}}\big)\Big)\prod_{n=1}^{k} p_n^{1/2^n},$$

donde $p_n$ es la precisión de $n$-gramas (coincidentes / presentes en la predicción) y el primer factor es la penalización por brevedad. Aquí usamos $k=2$. Vale 1 si la predicción es idéntica a la referencia.

In [ ]:
def bleu(pred, ref, k=2):
    p, r = pred.split(), ref.split()
    if not p:
        return 0.0
    score = math.exp(min(0, 1 - len(r) / len(p)))
    for n in range(1, k + 1):
        if len(p) < n:
            return 0.0
        ref_counts = collections.Counter(tuple(r[i:i + n]) for i in range(len(r) - n + 1))
        matches = 0
        for i in range(len(p) - n + 1):
            g = tuple(p[i:i + n])
            if ref_counts[g] > 0:
                matches += 1
                ref_counts[g] -= 1
        score *= (matches / (len(p) - n + 1)) ** (0.5 ** n)
    return score

print(bleu("a b b c d", "a b c d e f", k=3), bleu("a b", "a b c d e f", k=2))   # ejemplos de clase

eval_pairs = random.Random(1).sample(test_pairs, 500)   # subconjunto fijo para comparar modelos

def mean_bleu(model, translate_fn, pairs=eval_pairs):
    return sum(bleu(translate_fn(model, en), fr) for en, fr in pairs) / len(pairs)

import time
t0 = time.time()
print(f"BLEU (k=2), búsqueda voraz : {mean_bleu(model, translate_greedy):.4f}  ({time.time() - t0:.0f} s)")
t0 = time.time()
print(f"BLEU (k=2), haz k=4        : {mean_bleu(model, lambda m, s: translate_beam(m, s, k=4)):.4f}  ({time.time() - t0:.0f} s)")

### Efecto del tamaño del haz

¿Cómo cambian la calidad (BLEU) y el tiempo con $k$? Recuerda que $k=1$ es la búsqueda voraz.

In [ ]:
beam_results = {}
for k in (1, 2, 4, 8):
    t0 = time.time()
    b = mean_bleu(model, lambda m, s: translate_beam(m, s, k=k), eval_pairs[:200])
    beam_results[k] = (b, time.time() - t0)
    print(f"k = {k}: BLEU = {b:.4f}, tiempo = {beam_results[k][1]:.1f} s (200 oraciones)")

## Atención de Bahdanau

El modelo simple comprime toda la entrada en un **único** estado $(\mathbf{H},\mathbf{C})$, igual para todos los pasos del decodificador. Para oraciones largas esto es un cuello de botella, y al producir cada palabra no todas las palabras de entrada son relevantes.

La atención de Bahdanau et al. (2014) calcula un contexto **distinto en cada paso**:

$$\mathbf{c}_{t'} = \sum_{t=1}^{T}\alpha(\mathbf{s}_{t'-1},\mathbf{h}_t)\,\mathbf{h}_t,\qquad
\alpha(\mathbf{q},\mathbf{k}_i)=\frac{\exp a(\mathbf{q},\mathbf{k}_i)}{\sum_j \exp a(\mathbf{q},\mathbf{k}_j)},\qquad
a(\mathbf{q},\mathbf{k}) = \mathbf{w}_v^{\top}\tanh(\mathbf{W}_q\mathbf{q}+\mathbf{W}_k\mathbf{k}).$$

* **Consulta** $\mathbf{q}$: el estado oculto previo del decodificador (última capa).
* **Claves y valores**: los estados del codificador $\mathbf{h}_1,\dots,\mathbf{h}_T$ en todos los pasos.
* El contexto $\mathbf{c}_{t'}$ se **concatena** con el embedding del token de entrada y entra a la LSTM del decodificador.
* Los pesos de las posiciones de relleno se anulan con una máscara antes del softmax.

El decodificador ahora debe avanzar paso a paso también durante el entrenamiento, porque la consulta de cada paso depende del estado del paso anterior.

In [ ]:
class AdditiveAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.W_q = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.W_k = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.w_v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, q, K, mask):
        # q: (B, H)   K (= V): (B, T, H)   mask: (B, T), True en las posiciones reales
        scores = self.w_v(torch.tanh(self.W_q(q).unsqueeze(1) + self.W_k(K))).squeeze(-1)   # (B, T)
        scores = scores.masked_fill(~mask, -1e9)
        weights = torch.softmax(scores, dim=-1)
        context = (weights.unsqueeze(-1) * K).sum(dim=1)                                    # (B, H)
        return context, weights


class AttnDecoderLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers=2, dropout=0.1, pad_id=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_id)
        self.attention = AdditiveAttention(hidden_dim)
        self.rnn = nn.LSTM(embedding_dim + hidden_dim, hidden_dim, num_layers, batch_first=True,
                           dropout=dropout if num_layers > 1 else 0)
        self.out = nn.Linear(hidden_dim, vocab_size)

    def step(self, y_prev, state, enc_out, mask):          # y_prev: (B,)
        h, c = state
        context, weights = self.attention(h[-1], enc_out, mask)       # consulta: última capa
        x = torch.cat([self.embedding(y_prev), context], dim=-1).unsqueeze(1)
        o, state = self.rnn(x, state)
        return self.out(o[:, 0]), state, weights


class Seq2SeqAttnLSTM(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder, self.decoder = encoder, decoder

    def start(self, src, lengths):
        enc_out, state = self.encoder(src, lengths)
        mask = torch.arange(src.size(1), device=src.device)[None, :] < lengths.to(src.device)[:, None]
        return state, enc_out, mask

    def step(self, y_prev, ctx):
        state, enc_out, mask = ctx
        logits, state, weights = self.decoder.step(y_prev, state, enc_out, mask)
        return logits, (state, enc_out, mask), weights

    def forward(self, src, lengths, tgt_in):
        ctx = self.start(src, lengths)
        outputs = []
        for t in range(tgt_in.size(1)):                     # teacher forcing, un paso a la vez
            logits, ctx, _ = self.step(tgt_in[:, t], ctx)
            outputs.append(logits)
        return torch.stack(outputs, dim=1)                  # (B, T', V)


def build_seq2seq_attn():
    enc = EncoderLSTM(en_tok.get_vocab_size(), EMB_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT, EN_PAD)
    dec = AttnDecoderLSTM(fr_tok.get_vocab_size(), EMB_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT, PAD)
    return Seq2SeqAttnLSTM(enc, dec).to(device)

model_attn = build_seq2seq_attn()
print(f"Parámetros entrenables: {sum(p.numel() for p in model_attn.parameters() if p.requires_grad):,}")
src, lengths, tgt = next(iter(train_loader))
print("logits:", tuple(model_attn(src.to(device), lengths, tgt[:, :-1].to(device)).shape))

In [ ]:
t0 = time.time()
hist_attn = fit(model_attn)
print(f"Tiempo de entrenamiento: {time.time() - t0:.0f} s")

### Pesos de atención

Traducimos una oración guardando los pesos $\alpha$ de cada paso del decodificador. En el mapa de calor, cada fila es un token generado (consulta) y cada columna un token de la entrada (clave); cada fila suma 1.

In [ ]:
import matplotlib.pyplot as plt

@torch.no_grad()
def translate_with_attention(model, sentence, max_len=MAX_LEN):
    model.eval()
    src, lengths = encode_source(sentence)
    ctx = model.start(src, lengths)
    y = torch.tensor([START], device=device)
    out, attn = [], []
    for _ in range(max_len):
        logits, ctx, w = model.step(y, ctx)
        y = logits.argmax(-1)
        attn.append(w[0].cpu())
        if y.item() == END:
            break
        out.append(y.item())
    return out, torch.stack(attn)

sentence = "i'm home ."
out_ids, attn = translate_with_attention(model_attn, sentence)
src_tokens = en_tok.encode(normalize(sentence)).tokens
out_tokens = [fr_tok.id_to_token(i) for i in out_ids] + ['[end]']

fig, ax = plt.subplots(figsize=(5, 3.5))
im = ax.imshow(attn.numpy(), cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(len(src_tokens)), src_tokens, rotation=60)
ax.set_yticks(range(len(out_tokens)), out_tokens)
ax.set_xlabel('entrada (claves)'); ax.set_ylabel('salida (consultas)')
fig.colorbar(im); plt.tight_layout(); plt.show()
print(sentence, '->', fr_tok.decode(out_ids))

## Comparación de los modelos

Reunimos la pérdida y la perplejidad en prueba, el BLEU (búsqueda voraz y en haz con $k=4$) sobre el mismo subconjunto de 500 pares de prueba y los parámetros de cada modelo.

In [ ]:
import pandas as pd

def summarize(name, model, hist):
    return {
        'parámetros': sum(p.numel() for p in model.parameters()),
        'pérdida prueba': hist[-1][1],
        'perplejidad prueba': math.exp(hist[-1][1]),
        'BLEU voraz': mean_bleu(model, translate_greedy),
        'BLEU haz (k=4)': mean_bleu(model, lambda m, s: translate_beam(m, s, k=4)),
    }

results = {
    'seq2seq LSTM': summarize('seq2seq LSTM', model, hist_plain),
    'seq2seq LSTM + atención': summarize('seq2seq LSTM + atención', model_attn, hist_attn),
}
df = pd.DataFrame(results).T
display(df.style.format({'parámetros': '{:,.0f}', 'pérdida prueba': '{:.4f}',
                         'perplejidad prueba': '{:.2f}', 'BLEU voraz': '{:.4f}', 'BLEU haz (k=4)': '{:.4f}'}))

In [ ]:
# Oraciones de prueba de distintos largos: ¿ayuda la atención en las largas?
long_pairs = sorted(test_pairs, key=lambda p: len(p[0].split()))[-300:]
short_pairs = [p for p in test_pairs if len(p[0].split()) <= 4][:300]
for name, m in [('sin atención', model), ('con atención', model_attn)]:
    print(f"{name}: BLEU oraciones cortas = {mean_bleu(m, translate_greedy, short_pairs):.4f}, "
          f"largas = {mean_bleu(m, translate_greedy, long_pairs):.4f}")

**Preguntas para discutir**

1. ¿Qué diferencia hay entre la pérdida de entrenamiento y la de prueba al final de cada entrenamiento? ¿Qué te dice sobre el sobreajuste? (El artículo original no separa datos de prueba.)
2. ¿Mejora la búsqueda en haz el BLEU respecto de la voraz? ¿Cómo cambian el BLEU y el tiempo al subir $k$? ¿Tiene sentido un $k$ muy grande?
3. ¿Ayuda la atención? Compara oraciones cortas y largas. ¿Qué observas en el mapa de calor? Ten presente que cada modelo se entrenó una sola vez y con pocas épocas.
4. **Sin máscara:** reentrena el modelo simple sin `ignore_index=PAD` en la pérdida. ¿Qué cambia?
5. **Sin empaquetado:** quita `pack_padded_sequence` del codificador. ¿Qué le pasa al estado final cuando la oración es mucho más corta que el largo del minibatch?
6. **Sin teacher forcing:** durante el entrenamiento, alimenta al decodificador con su propia predicción previa (como hace el artículo). ¿Cómo cambian la velocidad de convergencia y el tiempo de cada época?
7. Reemplaza la LSTM por una GRU (`nn.GRU` no tiene estado de celda: `(h, c)` pasa a ser solo `h`).
8. Reemplaza la atención aditiva por **producto punto escalado** $\mathbf{q}^\top\mathbf{k}/\sqrt{d}$. ¿Cómo cambia el entrenamiento?
9. Aumenta `VOCAB_SIZE`, `NUM_LAYERS` o `EPOCHS`. ¿Hasta dónde mejora el BLEU?